# **LangChain Chaining**

In [4]:
!pip install langchain==0.3.28 langchain-google-genai langchain-community langchain-core numpy==1.26.4 wikipedia  ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 105.0 MB/s eta 0:00:00


In [2]:
!pip install grandalf

In [3]:
import os
import getpass
from langchain.llms import Ollama
from langchain_google_genai import ChatGoogleGenerativeAI

In [4]:
pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 kB 48.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.84
    Uninstalling langchain-core-0.3.84:
      Successfully uninstalled langchain-core-0.3.84
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.28 requires langchain-core<1.0.0,>=0.3.73, but you have langchain-core 1.2.30 which is incompatible.


In [2]:
pip install langchain-groq

  Using cached langchain_groq-1.1.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.9 MB/s eta 0:00:00


In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # or "mixtral-8x7b-32768", "gemma2-9b-it"
    api_key="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",       # or set GROQ_API_KEY env variable
    temperature=0.7,
)

# Simple Chain

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


prompt = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)

# model = ChatOpenAI()
model = llm

parser = StrOutputParser()

chain = prompt | model | parser

result = chain.invoke({'topic':'cricket'})

print(result)
chain.get_graph().print_ascii()

Here are five interesting facts about cricket:

1. **The oldest cricket club**: The Artillery Ground in London is considered the oldest cricket club in the world, with records of matches dating back to 1730. However, the Marylebone Cricket Club (MCC), founded in 1787, is the most prestigious and influential cricket club, responsible for creating the official rules of the game.

2. **The longest cricket match**: The longest cricket match in history was played between England and South Africa in 1939, lasting for 14 days. The match, which was a Test match, ended in a draw due to weather conditions and the onset of World War II.

3. **The fastest bowler**: The fastest recorded delivery in cricket was bowled by Pakistani paceman Aaqib Javed, who reached a speed of 161.3 km/h (100.2 mph) in 1993. However, the fastest average bowling speed is held by Australian bowler Shaun Tait, who averaged 157.3 km/h (97.8 mph) throughout his career.

4. **The first cricket World Cup**: The first Cricket 

# Sequential Chain

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt1 = PromptTemplate(
    template='Generate a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a 5 pointer summary from the following text \n {text}',
    input_variables=['text']
)

model =llm

parser = StrOutputParser()

chain = prompt1 | model | parser | prompt2 | model | parser

result = chain.invoke({'topic': 'Unemployment in India'})

print(result)
chain.get_graph().print_ascii()

Here is a 5-pointer summary of the report on unemployment in India:

1. **Unemployment Rate**: The unemployment rate in India stood at 7.2% in 2020-21, with approximately 30 million people being unemployed out of a total workforce of around 430 million, with higher rates in urban areas (9.3%) compared to rural areas (6.5%).

2. **Key Trends**: Unemployment is particularly high among young people (17.4%), women (10.3%), and in the manufacturing sector (10.5%), highlighting the need for targeted interventions to address these disparities and create more job opportunities in sectors such as manufacturing, services, and infrastructure.

3. **Causes of Unemployment**: The main causes of unemployment in India include demographic challenges (rapid population growth), slowdown in economic growth, lack of skills and education, and informalization of the economy, which can be addressed through initiatives such as job creation, skills development, and social protection.

4. **Consequences of Unem

# Parallel Chain

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

model1 = llm

model2 = llm

prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain = prompt3 | model1 | parser

chain = parallel_chain | merge_chain

text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

result = chain.invoke({'text':text})

print(result)
chain.get_graph().print_ascii()

**Notes on Support Vector Machines (SVMs)**

**Introduction to Support Vector Machines**

Support vector machines are used for classification, regression, and outliers detection. They have several advantages and disadvantages that are essential to understand for optimal performance.

**Advantages:**

1. Effective in high dimensional spaces.
2. Memory efficient.
3. Versatile with different kernel functions.

**Disadvantages:**

1. May over-fit if not regularized.
2. Does not provide direct probability estimates.

**Key Points:**

1. Can handle dense and sparse input data.
2. Requires specific data type for optimal performance (float64).

**Frequently Asked Questions**

1. **Q: What are support vector machines used for?**
   A: Support vector machines are used for classification, regression, and outliers detection.

2. **Q: What is an advantage of support vector machines in high dimensional spaces?**
   A: Support vector machines are effective in high dimensional spaces.

3. **Q: Why are

# Conditonal Chain

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser


parser = StrOutputParser()

class Feedback(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

print(chain.invoke({'feedback': 'This is a beautiful phone'}))

chain.get_graph().print_ascii()

Thank you so much for your kind words. I'm thrilled to hear that you're happy with the experience. Your positive feedback is greatly appreciated and I'm glad I could make a difference. If you have any other questions or need anything else, please don't hesitate to reach out.
    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
      +----------+       
      | ChatGroq |       
      +----------+       
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *         